## DEStatements - Derivative Notebook
Joins primaryactive.estatements with primary.exposure to add CaseCode and RelationType.

In [0]:
from typing import Any

from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

import os
import logging

import sys
sys.path.append("../aws_utils")
import aws_utils

env = aws_utils.get_env_prefix()
spark.conf.set("spark.databricks.io.cache.enabled", "false")

In [0]:
import delta_table_creator, transform

schema = (
    "DEStatementId string,"
    "EStatementId string,"
    "EStatementKey string,"
    "SourceSystem string,"
    "PortfolioKey string,"
    "PortfolioName string,"
    "IngestionTimestamp timestamp,"
    "ExposureKeyWithoutPrefix string,"
    "CDI string,"
    "TaxRegistrationNumber string,"
    "Email string,"
    "CaseCode string,"
    "RelationType string,"
    "OptIn string,"
    "DeceasedFlag boolean,"
    "Company string,"
    "StatementProductionMonth string,"
    "StatementProductionYear string,"
    "StagingKey string,"
    "StagingDateInserted timestamp,"
    "Filename string,"
    "ErrorCode string,"
    "ErrorMessage string,"
    "BusinessDate date"
)

partition_columns = ["BusinessDate"]
run_date = dbutils.widgets.get("run_date")
unity = dbutils.widgets.get("unity")
table_name = "destatements"
derivative_table_path = f"`{env}-cdh`.derivativeactive.{table_name}"

# ================== Create Delta Table ==================
logging.info("Creating Delta Table...")
if spark.catalog.tableExists(derivative_table_path) is False:
    delta_table_creator.create_delta_table(
        spark,
        derivative_table_path,
        partition_columns,
        schema,
    )
else:
    logging.info("Table already exists.")

# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    derivative_table_path,
    run_date,
)

In [0]:
# ================== Load primaryactive.estatements ==================

estatements = (
    spark.read.table(f"`{env}-cdh`.primaryactive.estatements")
    .filter(f"BusinessDate = '{run_date}'")
)

# ================== Load primary.exposure ==================

exposure = (
    spark.read.table(f"`{env}-cdh`.primary.exposure")
    .filter(f"BusinessDate = '{run_date}'")
    .select(
        col("ExposureKey").alias("exp_ExposureKey"),
        col("CDI").alias("exp_CDI"),
        "CaseCode",
        col("PartyRelationshipType").alias("RelationType"),
    )
)

In [0]:
# ========================== Join ==========================

exposure_cond = (
    (estatements.ExposureKeyWithoutPrefix == exposure.exp_ExposureKey)
    & (estatements.CDI == exposure.exp_CDI)
)

result_df = estatements.join(exposure, exposure_cond, "left")

# ========================== Select Final Columns ==========================

result_df = result_df.select(
    sha2(concat_ws(";", estatements.ExposureKeyWithoutPrefix, estatements.CDI, estatements.StagingKey, lit("extfile")), 256).alias("DEStatementId"),
    estatements.EStatementId,
    estatements.EStatementKey,
    estatements.SourceSystem,
    estatements.PortfolioKey,
    estatements.PortfolioName,
    from_utc_timestamp(current_timestamp(), "Europe/Athens").alias("IngestionTimestamp"),
    estatements.ExposureKeyWithoutPrefix,
    estatements.CDI,
    estatements.TaxRegistrationNumber,
    estatements.Email,
    exposure.CaseCode.cast("string").alias("CaseCode"),
    exposure.RelationType.cast("string").alias("RelationType"),
    estatements.OptIn,
    estatements.DeceasedFlag,
    estatements.Company,
    estatements.StatementProductionMonth,
    estatements.StatementProductionYear,
    estatements.StagingKey,
    estatements.StagingDateInserted,
    estatements.Filename,
    estatements.ErrorCode,
    estatements.ErrorMessage,
    estatements.BusinessDate,
)

# ========================== Write ==========================

result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
    derivative_table_path
)

logging.info("DEStatements derivative load complete.")